Steps:
1. Import packages needed
2. Input data (includes train test split, normalize)
3. Create model
4. Acquire parameters / Evaluate model
5. Hyperparameter analysis
6. Visualizations (GridSearch Heatmap, Scatterplot with best Gridsearch)

## 1. Import Packages

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
np.set_printoptions(precision=5)
random_state = 42
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge
from sklearn.model_selection import validation_curve
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    auc,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV

## 2. Input Data

In [19]:
mtgjson_data_df = pd.read_csv('data/finalCards.csv')
#mtgjson_data_df.astype({'text':'str'}).dtypes
mtgjson_data_df.head()

C:\Users\jaymj\AppData\Local\Temp\ipykernel_4380\857232422.py:1: DtypeWarning: Columns (0,1,2,3,7,8,9,10,11,12,13,14,16,18,20,21,22,23,24,26,27,29,30,31,32,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57) have mixed types. Specify dtype option on import or set low_memory=False.
  mtgjson_data_df = pd.read_csv('data/finalCards.csv')


,uuid,cardName,availability,colorIdentity,defense,edhrecRank,edhrecSaltiness,finishes,isAlternative,isGameChanger,...,contains: toughness,contains: trample,contains: turn,contains: unless,contains: untap,contains: upkeep,contains: value,contains: vigilance,contains: way,contains: white
0,00010d56-fe38-5e35-8aed-518019aa36a5,Sphinx of the Final Word,paper,U,NaN,10186.0,0.14,foil,False,False,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0001e0d0-2dcd-5640-aadc-a84765cf5fc9,Goblin King,paper,R,NaN,3454.0,0.34,nonfoil,False,False,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0003caab-9ff5-5d1a-bc06-976dd0457f19,Caravan Vigil,"mtgo, paper",G,NaN,12489.0,0.08,"nonfoil, foil",False,False,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0003d249-25d9-5223-af1e-1130f09622a7,Deadshot Minotaur,"mtgo, paper","G, R",NaN,25150.0,0.20,"nonfoil, foil",False,False,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0004822c-c181-5564-808d-a6cc48359a1a,Clone Legion,"arena, mtgo, paper",U,NaN,3488.0,0.48,"nonfoil, foil",False,False,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Add Lemmatization Step for cleaning text

In [21]:
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer

# download libraries
nltk.download('wordnet')
nltk.download('punkt')

def lemmatize_sentence(sentence):
    lemmatizer = WordNetLemmatizer()
    #Should words be lemmatized? (Have to lemmatize words based on noun and verb...)
    tokens = word_tokenize(sentence)
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return ''.join(lemmatized_tokens)

#lemmatize mtgjson avility text
#mtgjson_data_df['text_lemmatized'] = mtgjson_data_df['text'].apply(lemmatize_sentence)

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\jaymj\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jaymj\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


### Count specific Lemmatized Text
#### Acquire the top 100 frequent words one-hot encode them

In [ ]:
#Acquire counts for each word (not including stop words)
vectorizer = CountVectorizer(max_features=100, stop_words='english')
model = vectorizer.fit_transform(mtgjson_data_df['text'].fillna(''))
ability_text = vectorizer.get_feature_names_out()
stop_words_vect = vectorizer.get_stop_words()
columns_text = ["contains: "+ item for item in ability_text]
words_df = pd.DataFrame(model.toarray(), columns=columns_text)
words_df.head(10)
final_df = pd.concat([mtgjson_data_df,words_df], axis = 1)

### Train Test Split MTGJSON Data

In [ ]:
X_columns = mtgjson_data_df.columns.to_list()
y_column = "price"
X_columns.remove(y_column)

X = mtgjson_data_df[X_columns].select_dtypes(['bool','float64','int64'])
y = mtgjson_data_df[y_column]

#Split Data
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(X, y, random_state=random_state)

#Normalize Data
scaler = StandardScaler().fit(X_train_raw)
X_train_norm = scaler.transform(X_train_raw)
X_test_norm = scaler.transform(X_test_raw)

## 3. Create Model

In [ ]:
#Create Ridge Regression Model with alpha as 1.0
alpha = 1.0
clf = Ridge(alpha=alpha)
clf.fit(X_train_norm, y_train_raw)

ValueError: Input X contains NaN.
Ridge does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

## 4. Acquire Parameters / Evaluate Metrics

In [ ]:
parameters = clf.get_params
y_predict = clf.predict(X_test_norm)
r2_score = clf.score(X_test_norm, y_test_raw)
print(r2_score)